<a href="https://colab.research.google.com/github/WilliamQD/financial-RAG/blob/main/Pinecone_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install \
    "pinecone" \
    "langchain-pinecone" \
    "langchain-openai" \
    "langchain-text-splitters" \
    "langchain" \
    "langchain-community" \
    "dropbox"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: aiohttp
    Found existing insta

In [2]:
import dropbox
import os
import requests
from google.colab import userdata
import pandas as pd
import io
import json

# ---------------------------
# SETUP DROPBOX API
# ---------------------------
def get_new_access_token():
    app_key = userdata.get('DbxAppKey')
    app_secret = userdata.get('DbxAppSecret')
    refresh_token = userdata.get('DbxRefreshToken')

    # Exchange the refresh token for a new access token
    token_url = "https://api.dropbox.com/oauth2/token"
    data = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "client_id": app_key,
        "client_secret": app_secret
    }

    response = requests.post(token_url, data=data)
    response.raise_for_status()  # Raise an exception if there's an error
    tokens = response.json()
    return tokens["access_token"]

def check_token_validity(access_token):
    # Example call to check if the token works by getting current account info
    check_url = "https://api.dropboxapi.com/2/users/get_current_account"
    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    response = requests.post(check_url, headers=headers)
    return response.status_code == 200

# ---------------------------
# GENERATE TOKEN AND PROCESS THE MERGED DIRECTORY FILE
# ---------------------------
db_access_token = get_new_access_token()

dbx = dropbox.Dropbox(db_access_token)
_, res = dbx.files_download('/data/merged.csv')
merged = pd.read_csv(io.BytesIO(res.content))

_, res = dbx.files_download('/data/sample/sample_1000gvkey_bybatch.csv')
test_df = pd.read_csv(io.BytesIO(res.content))

In [52]:
from google.colab import userdata
import os
from pinecone import Pinecone
from openai import OpenAI
from typing import Optional
import numpy as np

# Import the PineconeEmbeddings from langchain_pinecone
from langchain_pinecone import PineconeEmbeddings

# ---------------------------
# PROMPT TEMPLATES
# ---------------------------
Q1_PROMPT_TEMPLATE = """
You are the CEO of {company_name}, identified by the gvkey number {gvkey}. The current year is {fyear}.
You will be provided several sections of information and based on these information, please perform the tasks at the end and provide detailed analyses.

=== QUARTER/ANNUAL REPORTS FOR {company_name} IN THE PAST YEARS ===
{item1_and_item7_text}
=== END OF REPORTS ===

=== CONFERENCE CALL EVENT TEXT BY {company_name} IN THE PAST YEARS ===
{conference_call_text}
=== END OF CONFERENCE CALL EVENT TEXT ===

=== PATENTS BY {company_name} IN THE PAST YEARS ===
{patents_text}
=== END OF PATENTS ===

=== WALL STREET JOURNAL FRONT PAGE ARTICLE TEXT IN THE PAST YEARS ===
{wsj_text}
=== END OF ARTICLE ===

=== QUESTION ===
{question}
=== END OF QUESTION ===
"""

Q2_PROMPT_TEMPLATE = """
Based on the chat history and the answer you made before, please perform the tasks at the end and provide detailed analyses.

=== CHAT HISTORY ===
{chat_history_str}
=== END OF CHAT HISTORY ===

=== QUESTION ===
{question}
=== END OF QUESTION ===
"""

QUESTION_1 = """
1. Read the Business Description Section: Describe the company’s core business and strategic direction.
2. Read the Management Discussion and Analysis: Analyze the MD&A section to grasp management’s interpretation of past performance, current challenges, and future outlook.
3. Identify Current Investment Focus: Determine the company’s current projects and business model as disclosed in the annual report.
4. Analyze the Market and Competitive Environment: Examine the industry dynamics and competitive environment mentioned in the annual report.

Please provide a detailed report summarizing your findings from these steps.
"""

QUESTION_2 = """
Based on your previous analysis, your next task is to predict the company’s next three potential projects for consideration.

Formatting Guidelines:
  - Return **only** JSON that matches the given schema.
  - Each object must include:
    “PROJECT”: The name of the proposed project.
    “DESCRIPTION”: A brief description of the project.
    “MARKET VALUE”: Estimated market value in million dollars USD (discounted present value of future cash flows).
    “IMPLEMENTATION COST”: Estimated cost to implement the project in million dollars USD (can be larger or smaller than the market value).
    “REASONING”: Explanation for why this project is proposed.
    “CONFIDENCE”: A confidence level in the prediction (0-100).
    “SIMILAR FIRMS”: A list of three public firms engaged in similar businesses, including their names and tickers.
    “PRIORITY”: A priority ranking (1-3). 1 is highest.
    “PRIORITY_REASONING”: Justification for the assigned priority.
"""

# ---------------------------
# SETUP PINECONE and OpenAI
# ---------------------------
pc = Pinecone(
    api_key= userdata.get('PineconeKey')
)
os.environ['PINECONE_API_KEY'] = userdata.get('PineconeKey')
index = pc.Index("finance-rag")

client = OpenAI(api_key=userdata.get("OpenAI4Pinecone"))

# ---------------------------
# SETUP EMBEDDING MODEL
# ---------------------------
embedding_model = PineconeEmbeddings(
    model='multilingual-e5-large')

def embed_query(query: str) -> list:
    """
    Embed the query using the PineconeEmbeddings model.
    Returns a list of floats representing the embedding vector.
    """
    return embedding_model.embed_query(query)

# ---------------------------
# RETRIEVAL FUNCTION
# ---------------------------
def retrieve_from_namespace(
    query: str,
    namespace: str,
    metadata_filter: Optional[dict] = None,
    top_k: int = 10
) -> str:
    """
    Retrieve text from a given Pinecone namespace using the specified query and metadata filter.
    Returns a numbered list string of the top matching documents' content.
    """
    query_vec = embed_query(query)
    result = index.query(
        vector=query_vec,
        namespace=namespace,
        filter=metadata_filter if metadata_filter else {},
        top_k=top_k,
        include_metadata=True
    )

    # 2) count how many matches you actually retrieved
    retrieved_count = len(result.get("matches", []))

    # 3) count how many vectors in that namespace satisfy your filter
    result_2 = index.query(
        vector=query_vec,
        namespace=namespace,
        filter=metadata_filter if metadata_filter else {},
        top_k=10000,
        include_metadata=True
    )

    filtered_count = len(result_2.get("matches", []))

    print(f"[{namespace}] retrieved {retrieved_count} / filtered {filtered_count}")

    if (retrieved_count != filtered_count) and retrieved_count < top_k:
      print("\n wrong retrieval \n")

    text_chunks = []
    for i, match in enumerate(result.get("matches", []), start=1):
        doc_text = match.get("metadata", {}).get("text", "")
        if doc_text:
            # Prepend each chunk with its index
            text_chunks.append(f"{i}. {doc_text}")

    # Join with newlines to form a numbered list
    return "\n".join(text_chunks)


# ---------------------------
# PROMPT BUILDERS
# ---------------------------
def build_Q1_prompt(
    company_name: str,
    gvkey: int,
    fyear: int,
    item1_and_item7_text: str,
    conference_call_text: str,
    patents_text: str,
    wsj_text: str,
    question: str
) -> str:
    """
    Populate the Q1 prompt template using retrieved text from multiple namespaces.
    """
    return Q1_PROMPT_TEMPLATE.format(
        company_name=company_name,
        gvkey=gvkey,
        fyear=fyear,
        item1_and_item7_text=item1_and_item7_text,
        conference_call_text=conference_call_text,
        patents_text=patents_text,
        wsj_text=wsj_text,
        question=question
    )

# for conference call inital query
def make_conf_call_query(company: str, year: int) -> str:
    """
    Build a focused conference-call query string.
    """
    return (
        f"{year} {company} conference call highlights: "
        "financial performance, key strategic priorities"
    )

# QUERY EACH NAMESPACE WITH CONFERENCE CALL
def make_namespaces_query(
    namespace: str,
    company: str,
    conference_call_text: str
) -> str:
    """
    Build a focused user query string for Pinecone retrieval in a given namespace,
    by asking the LLM to distill the conference-call transcript into one concise query.
    """
    # Human-friendly name for the prompt
    namespace_text = {
        "10K-item1": "10-K MD&A and Business Description",
        "10K-item7": "10-K Financial Notes",
        "patents":   "Patent Filings",
        "wsj_frontpage": "Wall Street Journal Articles"
    }.get(namespace, namespace)

    # Craft the LLM prompt
    prompt_llm = f"""
    You are an expert in financial research. Based on the following {company}
    conference-call excerpts, generate two optimal search queries that
    will help match the most relevant document chunks from the “{namespace_text}” index.
    You should consider cosine similarity of chunks matching.

    Conference‑call material:
    \"\"\"
    {conference_call_text}
    \"\"\"

    Output just the query (no explanation and any additional sentences).
    """
    if not conference_call_text.strip():
        prompt_llm = f"""
        You are an expert in financial research. Based on your knowledge about {company},
        generate two optimal search queries that will help match the most relevant
        document chunks from the “{namespace_text}” index. You should consider cosine similarity of chunks matching.
        Output just the query (no explanation and any additional sentences).
        """

    # Call the LLM
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_llm}],
        max_tokens=256,
        temperature=0.0
    )
    # Grab and return the generated query
    return resp.choices[0].message.content.strip()


In [6]:
# STRUCTUED OUTPUT CLASSES

from typing import List, Optional
from pydantic import BaseModel, Field, constr

class Firm(BaseModel):
    name: str
    ticker: constr(strip_whitespace=True, to_upper=True)

class Project(BaseModel):
    project: str
    description: str
    market_value: float = Field(..., description="$ MM")
    implementation_cost: float = Field(..., description="$ MM")
    reasoning: str
    confidence: int
    similar_firms: List[Firm]
    priority: int
    priority_reasoning: str

class ProjectsPayload(BaseModel):
    projects: List[Project]

In [18]:
# ---------------------------
# HELPER FUNCTION TO EXTRACT Qs
# ---------------------------
import re
import json
from typing import Tuple, Optional, List

def extract_q_values(projects: List[Project]) -> Tuple[Optional[float], Optional[float], Optional[float]]:
    """
    Given a list of Project objects, return the three q‑ratios:
      q1 = projects[0].market_value / projects[0].implementation_cost
      q2 = projects[1].market_value / projects[1].implementation_cost
      q3 = projects[2].market_value / projects[2].implementation_cost

    If there are fewer than three projects or any error occurs, returns (None, None, None).
    """
    if not projects or len(projects) < 3:
        return None, None, None

    try:
        q1 = projects[0].market_value / projects[0].implementation_cost
        q2 = projects[1].market_value / projects[1].implementation_cost
        q3 = projects[2].market_value / projects[2].implementation_cost
    except Exception:
        return None, None, None

    return q1, q2, q3

In [50]:
import json
import re
from typing import Tuple, Dict, Any

# MAIN RUNNING FUNCTION FOR LLM
def generate_predictions(
    gvkey: int,
    fyear: int,
    cusip: str,
    comn: str,
    client
) -> Dict[str, Any]:
    """
    1. Seeds retrieval from conference calls
    2. Generates per‑namespace queries based on those calls
    3. Retrieves 10-K, patents, and WSJ text
    4. Builds Q1 & Q2 prompts and calls the LLM
    5. Extracts q1, q2, q3 ratios
    6. Returns a dict of all results
    """
    gvkey_str = str(gvkey)

    # PART 1: conference‑call seed & namespace queries
    conf_call_query = make_conf_call_query(comn, fyear)
    conf_call_filter = {"fiscal_year": {"$lte": fyear}, "CUSIP": cusip}
    text_conf_call = retrieve_from_namespace(
        query=conf_call_query,
        namespace="conference_call",
        metadata_filter=conf_call_filter,
        top_k=200
    )

    # Build one query per namespace
    namespaces = ["10K-item1", "10K-item7", "patents", "wsj_frontpage"]
    part1_queries: Dict[str, str] = {}
    for ns in namespaces:
        part1_queries[ns] = make_namespaces_query(ns, comn, text_conf_call)

    # PART 2: retrieve from each namespace
    item1_text = retrieve_from_namespace(
        query=part1_queries["10K-item1"],
        namespace="10K-item1",
        metadata_filter={"fyear": {"$lte": fyear}, "gvkey": gvkey},
        top_k=200
    )

    item7_text = retrieve_from_namespace(
        query=part1_queries["10K-item7"],
        namespace="10K-item7",
        metadata_filter={"fyear": {"$lte": fyear}, "gvkey": gvkey},
        top_k=200
    )
    combined_10k = "\n----\n".join([item1_text, item7_text])

    patents_text = retrieve_from_namespace(
        query=part1_queries["patents"],
        namespace="patents",
        metadata_filter={"filing_year": {"$lte": fyear}, "gvkey": gvkey_str},
        top_k=100
    )
    wsj_text = retrieve_from_namespace(
        query=part1_queries["wsj_frontpage"],
        namespace="wsj_frontpage",
        metadata_filter={"year": {"$lte": fyear}},
        top_k=10
    )

    # PART 3: Q1
    q1_prompt = build_Q1_prompt(
        company_name=comn,
        gvkey=gvkey,
        fyear=fyear,
        item1_and_item7_text=combined_10k,
        conference_call_text=text_conf_call,
        patents_text=patents_text,
        wsj_text=wsj_text,
        question=QUESTION_1
    )
    resp1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": q1_prompt}],
        max_tokens=1024,
        temperature=0.0
    )
    answer_q1 = resp1.choices[0].message.content.strip()

    # PART 4: Q2
    chat_hist = f"User asked Q1: {QUESTION_1}\nAssistant answered: {answer_q1}"
    q2_prompt = Q2_PROMPT_TEMPLATE.format(
        chat_history_str=chat_hist,
        question=QUESTION_2
    )
    resp2 = client.responses.parse(
        model="gpt-4o-mini",
        input=[{"role": "user", "content": q2_prompt}],
        max_output_tokens=1500,
        temperature=0.0,
        text_format=ProjectsPayload
    )
    answer_q2_projects = resp2.output_parsed.projects
    answer_q2 = json.loads(resp2.output_parsed.model_dump_json())

    # PART 5: extract q1,q2,q3
    q1, q2, q3 = extract_q_values(answer_q2_projects)

    part1_queries_str = "\n\n".join(f"{ns}: {qry}" for ns, qry in part1_queries.items())

    return {
        "gvkey": gvkey,
        "fyear": fyear,
        "part1_queries": part1_queries_str,
        "q1_prompt": q1_prompt,
        "q1_answer": answer_q1,
        "q2_answer": answer_q2,
        "q1": q1,
        "q2": q2,
        "q3": q3
    }

In [54]:
from tqdm import tqdm

# ---------------------------
# RUN TEST SAMPLES
# ---------------------------

cols = ['gvkey', 'fyear', 'part1_queries', 'q1_prompt', 'q1_answer', 'q2_answer', 'q1', 'q2', 'q3']
dtypes = {
    'gvkey': 'int64',
    'fyear': 'int64',
    'part1_queries': 'object',
    'q1_prompt': 'object',
    'q1_answer': 'object',
    'q2_answer': 'object',
    'q1':    'float64',
    'q2':    'float64',
    'q3':    'float64'
}
result = pd.DataFrame(columns=cols)
result = result.astype(dtypes)


sample_size = 10
sub_sample = test_df[test_df['sample'] == sample_size]
gvkeys = sub_sample['gvkey'].astype(str).str.lstrip('0').astype(int)

df_filtered = merged[
    merged['gvkey'].isin(gvkeys) &
    merged['fyear'].between(1993, 2022)
]

# MAIN RUNNING LOOP
first_n = df_filtered


for idx, row in tqdm(first_n.iterrows(), total=first_n.shape[0]):
    gvkey = row['gvkey']
    fyear = row['fyear']
    cusip = row['cusip']
    company_name = row['conm']

    output_row = generate_predictions(
        gvkey=gvkey,
        fyear=fyear,
        cusip=cusip,
        comn=company_name,
        client=client
    )
    result.loc[len(result)] = output_row

  0%|          | 0/218 [00:00<?, ?it/s]

[conference_call] retrieved 0 / filtered 0
[10K-item1] retrieved 0 / filtered 0
[10K-item7] retrieved 0 / filtered 0
[patents] retrieved 0 / filtered 0
[wsj_frontpage] retrieved 10 / filtered 10000


  0%|          | 0/218 [00:24<?, ?it/s]


KeyboardInterrupt: 

In [32]:
result

,gvkey,fyear,part1_queries,q1_prompt,q1_answer,q2_answer,q1,q2,q3
0,5523,1993,"10K-item1: 1. ""HAVERTY FURNITURE 10-K MD&A Bus...","\nYou are the CEO of HAVERTY FURNITURE, identi...","Based on the provided information, I will summ...",[project='E-Commerce Expansion' description='D...,3.000,3.333333,4.000000
1,8606,1993,"10K-item1: 1. ""Pitney Bowes Inc 10-K MD&A Busi...","\nYou are the CEO of PITNEY BOWES INC, identif...",### Detailed Report on Pitney Bowes Inc. (1993...,[project='Smart Mailing Solutions' description...,1.875,2.000000,2.000000
2,3157,1994,"10K-item1: 1. ""Coherent Inc 10-K MD&A Business...","\nYou are the CEO of COHERENT INC, identified ...","Based on the provided information, I will summ...",[project='Advanced Medical Laser Systems' desc...,1.875,1.714286,1.666667
3,4186,1994,"10K-item1: 1. ""EASTERN CO 10-K MD&A Business D...","\nYou are the CEO of EASTERN CO, identified by...","Based on the provided information, I will summ...",[project='Advanced Fastening Technology Develo...,3.000,3.000000,3.333333
4,5523,1994,"10K-item1: 1. ""HAVERTY FURNITURE 10-K MD&A Bus...","\nYou are the CEO of HAVERTY FURNITURE, identi...","Based on the provided information, I will summ...",[project='E-Commerce Expansion' description='E...,3.000,3.333333,3.000000


In [33]:
result.to_csv("test_results.csv", index=False)